# Learning curves from completed training runs

This notebook visualises artifacts written by the training notebooks.
It does not load the dataset, train a model, select a threshold, or run
an inference benchmark.

For the CNN, the stored epoch history provides train/validation loss and
accuracy. For the legacy MLP, scikit-learn stores only the training-loss
curve; validation was evaluated after the single `fit()` call and is not
an epoch-level training history. If the separate PyTorch MLP has been
trained, this notebook also plots its loss and accuracy history.


In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


WORK_ROOT = Path.cwd().resolve()

DATA_FOLDER = Path("/hercules/results/akazantsev/rfim_dataset")
META_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels_meta.csv"
SPLIT_PATH = DATA_FOLDER / "split_indices.npz"
PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels.npy"

# The training subset deliberately retains only statistical features and labels.
# These full files retain channel and segment identity and are used only by the
# inference-timing notebook, where one input must correspond to a real 256-channel
# observation segment.
FULL_META_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels_meta.csv"
FULL_PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels.npy"
SUBSET_SOURCE_INDICES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_indices.npy"

# Change the tag only for a deliberate new experiment. Existing results are never overwritten.
RUN_TAG = "b0531_legacy_performance_v1"
RUN_ROOT = WORK_ROOT / "outputs" / "performance_comparison" / RUN_TAG


def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def write_json(path: Path, payload: dict) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(payload), handle, indent=2, sort_keys=True)
        handle.write("\n")


def git_revision() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=WORK_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


In [ ]:
import joblib
import matplotlib.pyplot as plt

cnn_cpu_dir = RUN_ROOT / "cnn_cpu_legacy_max"
cnn_gpu_dir = RUN_ROOT / "cnn_cuda_legacy_max"
mlp_dir = RUN_ROOT / "mlp_orig_top3"
figure_dir = RUN_ROOT / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)


def find_cnn_history(run_dir: Path, device_name: str) -> Path:
    # Supports both the standard name and the device-suffixed name.
    candidates = [
        run_dir / "history.csv",
        run_dir / f"history_{device_name}.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"No history CSV was found in {run_dir}. Expected one of: {candidates}"
    )


histories = {
    "CPU": pd.read_csv(find_cnn_history(cnn_cpu_dir, "cpu")),
    "GPU": pd.read_csv(find_cnn_history(cnn_gpu_dir, "cuda")),
}
for device_name, history in histories.items():
    required = {
        "epoch", "train_loss", "validation_loss", "train_accuracy", "validation_accuracy"
    }
    missing = required.difference(history.columns)
    if missing:
        raise ValueError(f"{device_name} history is missing columns: {sorted(missing)}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.0), sharex="col")

for row, (device_name, history) in enumerate(histories.items()):
    ax_loss, ax_accuracy = axes[row]
    epoch = history["epoch"]

    ax_loss.plot(epoch, history["train_loss"], label="train", color="#4C78A8")
    ax_loss.plot(epoch, history["validation_loss"], label="validation", color="#E45756")
    ax_loss.set_title(f"1D CNN {device_name}: loss")
    ax_loss.set_ylabel("BCEWithLogitsLoss")
    ax_loss.grid(linestyle="--", alpha=0.35)
    ax_loss.legend(frameon=False)

    ax_accuracy.plot(epoch, history["train_accuracy"], label="train", color="#4C78A8")
    ax_accuracy.plot(epoch, history["validation_accuracy"], label="validation", color="#E45756")
    ax_accuracy.set_title(f"1D CNN {device_name}: accuracy")
    ax_accuracy.set_ylabel("Accuracy at threshold 0.5")
    ax_accuracy.set_ylim(0, 1.02)
    ax_accuracy.grid(linestyle="--", alpha=0.35)
    ax_accuracy.legend(frameon=False)

axes[1, 0].set_xlabel("Epoch")
axes[1, 1].set_xlabel("Epoch")
fig.suptitle("Learning curves of the legacy 1D CNN", y=1.01)
fig.tight_layout()
fig.savefig(figure_dir / "cnn_learning_curves_cpu_gpu.png", dpi=300, bbox_inches="tight")
fig.savefig(figure_dir / "cnn_learning_curves_cpu_gpu.pdf", bbox_inches="tight")
plt.show()


In [ ]:
mlp_bundle_path = mlp_dir / "mlp_orig_top3.joblib"
if not mlp_bundle_path.exists():
    raise FileNotFoundError(f"Run the MLP training notebook first: {mlp_bundle_path}")

mlp_bundle = joblib.load(mlp_bundle_path)
classifier = mlp_bundle["pipeline"].named_steps["clf"]
loss_curve = np.asarray(classifier.loss_curve_, dtype=float)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(np.arange(1, len(loss_curve) + 1), loss_curve, color="#4C78A8")
ax.set(
    xlabel="Training iteration",
    ylabel="Training loss",
    title="MLP (mean_o, std_o, skew_o): training loss",
)
ax.grid(linestyle="--", alpha=0.35)
fig.tight_layout()
fig.savefig(figure_dir / "mlp_training_loss.png", dpi=300, bbox_inches="tight")
fig.savefig(figure_dir / "mlp_training_loss.pdf", bbox_inches="tight")
plt.show()

print(f"Saved learning-curve figures in {figure_dir}")


In [ ]:
pytorch_mlp_history_path = RUN_ROOT / "mlp_pytorch_orig_top3" / "history.csv"
if not pytorch_mlp_history_path.exists():
    print("PyTorch MLP history is not available yet; skip its learning-curve figure.")
else:
    history = pd.read_csv(pytorch_mlp_history_path)
    required = {"epoch", "train_loss", "validation_loss", "train_accuracy", "validation_accuracy"}
    missing = required.difference(history.columns)
    if missing:
        raise ValueError(f"PyTorch MLP history is missing columns: {sorted(missing)}")

    fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0))
    axes[0].plot(history["epoch"], history["train_loss"], label="train", color="#4C78A8")
    axes[0].plot(history["epoch"], history["validation_loss"], label="validation", color="#E45756")
    axes[0].set(xlabel="Epoch", ylabel="BCEWithLogitsLoss", title="PyTorch MLP: loss")
    axes[0].grid(linestyle="--", alpha=0.35)
    axes[0].legend(frameon=False)

    axes[1].plot(history["epoch"], history["train_accuracy"], label="train", color="#4C78A8")
    axes[1].plot(history["epoch"], history["validation_accuracy"], label="validation", color="#E45756")
    axes[1].set(
        xlabel="Epoch", ylabel="Accuracy at threshold 0.5", title="PyTorch MLP: accuracy", ylim=(0, 1.02)
    )
    axes[1].grid(linestyle="--", alpha=0.35)
    axes[1].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(figure_dir / "mlp_pytorch_learning_curves.png", dpi=300, bbox_inches="tight")
    fig.savefig(figure_dir / "mlp_pytorch_learning_curves.pdf", bbox_inches="tight")
    plt.show()
